<a href="https://colab.research.google.com/github/ZiqiLi379/STATS-302-Intro-to-ML/blob/main/STATS302_Week_3_Learning_PyTorch_with_Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Week 3 - Learning PyTorch with Examples
###STATS 302 Principle of Machine Learning
###Duke Kunshan University

This tutorial introduces the fundamental concepts of PyTorch through self-contained examples.

At its core, PyTorch provides two main features:
* An n-dimensional Tensor, similar to numpy but can run on GPUs
* Automatic differentiation for building and training neural networks

We will use a problem of fitting y=sin(x) with a third order polynomial as our running example. The network will have four parameters, and will be trained with gradient descent to fit random data by minimizing the Euclidean distance between the network output and the true output.

The "forward pass" refers to calculation process, values of the output layers from the inputs data. It's traversing through all neurons from first to last layer.

A "loss function" is calculated from the output values.

And then "backward pass" refers to process of counting changes in weights (de facto learning), using gradient descent algorithm (or similar). Computation is made from last layer, backward to the first layer.

Backward and forward pass makes together one "iteration".



Warm-up: numpy
--------------

A third order polynomial, trained to predict $y=\sin(x)$ from $-\pi$
to $\pi$ by minimizing squared Euclidean distance.

This implementation uses numpy to manually compute the forward pass, loss, and
backward pass.

A numpy array is a generic n-dimensional array; it does not know anything about
deep learning or gradients or computational graphs, and is just a way to perform
generic numeric computations.



In [7]:
import numpy as np
import math

# Create random input and output data
x = np.linspace(-math.pi, math.pi, 2000)
y = np.sin(x)

# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    # y = a + b x + c x^2 + d x^3
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = np.square(y_pred - y).sum()

    # Backward pass: compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)  # \partial L/\partial y_pred_i
    grad_a = grad_y_pred.sum()  # Use chain rule, \partial L/\partial a
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()
    if t % 100 == 99:
        print(t, loss, grad_a, grad_b, grad_c, grad_d)

    # Update weights
    a -= learning_rate * grad_a  # Gradient descent
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

99 968.681023991443 371.4805318424401 -1929.467085145278 -50.94557185890528 274.4498909147137
199 645.4130467478348 310.64620793538666 -1568.9908105167178 -53.58861581888698 223.17527990381734
299 431.10122601618275 261.3220589843731 -1275.861185940133 -45.082402896650024 181.48014340302723
399 288.9972280031184 219.8299161018117 -1037.4960483372565 -37.92431851299608 147.5747782809002
499 194.75476234511885 184.92580466683717 -843.6639206343333 -31.90277848370198 120.00384602017596
599 132.24148334381758 155.56369141242465 -686.0448404799336 -26.83732535964134 97.58390442723652
699 90.76626980798997 130.8636300350779 -557.8732379538762 -22.57615376125358 79.35261009603792
799 63.24291222446171 110.08539017344202 -453.64753331199364 -18.991561633719442 64.52741121615432
899 44.9738330042119 92.6062736177393 -368.893989672025 -15.97612317410389 52.47195767371741
999 32.84439356798667 77.90245281278472 -299.9746843604114 -13.439469412615907 42.66878664772659
1099 24.789119457878446 65.53

In [6]:
print(x, x**2)

[-3.14159265 -3.13844949 -3.13530633 ...  3.13530633  3.13844949
  3.14159265] [9.8696044  9.8498652  9.83014575 ... 9.83014575 9.8498652  9.8696044 ]



PyTorch: Tensors
----------------

A third order polynomial, trained to predict $y=\sin(x)$ from $-\pi$
to $\pi$ by minimizing squared Euclidean distance.

This implementation uses PyTorch tensors to manually compute the forward pass,
loss, and backward pass.

A PyTorch Tensor is basically the same as a numpy array: it does not know
anything about deep learning or computational graphs or gradients, and is just
a generic n-dimensional array to be used for arbitrary numeric computation.

The biggest difference between a numpy array and a PyTorch Tensor is that
a PyTorch Tensor can run on either CPU or GPU. To run operations on the GPU,
just cast the Tensor to a cuda datatype.


In [8]:
import torch
print(torch.__version__)
torch.cuda.is_available()

2.5.1


False

In [13]:
import torch
import math

dtype = torch.float
device = torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU (NVIDIA Only!!!)

# Create random input and output data
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum().item()  # Use .item() to convert to Python float
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 69.14936065673828
199 48.7752685546875
299 35.2848014831543
399 26.350969314575195
499 20.434032440185547
599 16.51490020751953
699 13.918649673461914
799 12.198562622070312
899 11.058816909790039
999 10.303487777709961
1099 9.802851676940918
1199 9.470972061157227
1299 9.250921249389648
1399 9.105000495910645
1499 9.008220672607422
1599 8.944009780883789
1699 8.901397705078125
1799 8.873116493225098
1899 8.854344367980957
1999 8.841876983642578
Result: y = -0.0014719945611432195 + 0.8521016240119934 x + 0.000253942736890167 x^2 + -0.0926705002784729 x^3


In [10]:
print(x)
print(y)

tensor([-3.1416, -3.1384, -3.1353,  ...,  3.1353,  3.1384,  3.1416])
tensor([ 8.7423e-08, -3.1430e-03, -6.2863e-03,  ...,  6.2863e-03,
         3.1430e-03, -8.7423e-08])
